Association Rules: The FP Growth Algorithm \
Sources: https://www.kaggle.com/code/rjmanoj/fp-growth-algorithm-frequent-itemset-pattern \
https://codinginfinite.com/implement-fp-growth-algorithm-in-python/

FP-Growth es un algoritmo que está diseñado para operar en bases de datos que contienen transacciones, como compras de clientes en una tienda. \
Un itemset se considera “frecuente” si cumple con un umbral de soporte definido por el usuario. \
Por ejemplo, si el umbral de soporte es 0.5 (50%), un conjunto frecuente es aquel que aparece en al menos el 50% de todas las transacciones de la base de datos.

In [2]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

In [25]:
dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

In [26]:
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)
df

,Apple,Corn,Dill,Eggs,Ice cream,Kidney Beans,Milk,Nutmeg,Onion,Unicorn,Yogurt
0,False,False,False,True,False,True,True,True,True,False,True
1,False,False,True,True,False,True,False,True,True,False,True
2,True,False,False,True,False,True,True,False,False,False,False
3,False,True,False,False,False,True,True,False,False,True,True
4,False,True,False,True,True,True,False,False,True,False,False


fpgrowth() function returns a dataframe containing the frequent itemsets and their support

In [27]:
from mlxtend.frequent_patterns import fpgrowth 

fpgrowth(df, min_support=0.6)

,support,itemsets
0,1.0,(5)
1,0.8,(3)
2,0.6,(10)
3,0.6,(8)
4,0.6,(6)
5,0.8,"(3, 5)"
6,0.6,"(10, 5)"
7,0.6,"(8, 3)"
8,0.6,"(8, 5)"
9,0.6,"(8, 3, 5)"


In [28]:
frequent_itemset = fpgrowth(df, min_support=0.6, use_colnames=True)
frequent_itemset

,support,itemsets
0,1.0,(Kidney Beans)
1,0.8,(Eggs)
2,0.6,(Yogurt)
3,0.6,(Onion)
4,0.6,(Milk)
5,0.8,"(Kidney Beans, Eggs)"
6,0.6,"(Kidney Beans, Yogurt)"
7,0.6,"(Onion, Eggs)"
8,0.6,"(Kidney Beans, Onion)"
9,0.6,"(Kidney Beans, Onion, Eggs)"


Generate association rules using the association_rules() function

In [35]:
from mlxtend.frequent_patterns import association_rules
import warnings

# Ignorar warnings de división inválida
warnings.filterwarnings("ignore", category=RuntimeWarning)

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.8, support_only=False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(3),(5),0.8,1.0,0.8,1.0,1.00,1.0,0.00,inf,0.0,0.80,0.0,0.900
1,(5),(3),1.0,0.8,0.8,0.8,1.00,1.0,0.00,1.0,0.0,0.80,0.0,0.900
2,(10),(5),0.6,1.0,0.6,1.0,1.00,1.0,0.00,inf,0.0,0.60,0.0,0.800
3,(8),(3),0.6,0.8,0.6,1.0,1.25,1.0,0.12,inf,0.5,0.75,1.0,0.875
4,(8),(5),0.6,1.0,0.6,1.0,1.00,1.0,0.00,inf,0.0,0.60,0.0,0.800
5,"(8, 3)",(5),0.6,1.0,0.6,1.0,1.00,1.0,0.00,inf,0.0,0.60,0.0,0.800
6,"(8, 5)",(3),0.6,0.8,0.6,1.0,1.25,1.0,0.12,inf,0.5,0.75,1.0,0.875
7,(8),"(3, 5)",0.6,0.8,0.6,1.0,1.25,1.0,0.12,inf,0.5,0.75,1.0,0.875
8,(6),(5),0.6,1.0,0.6,1.0,1.00,1.0,0.00,inf,0.0,0.60,0.0,0.800


Si las reglas incluyen ítems muy raros (que generan denominador 0), se puede modificar el min_support al calcular los itemsets frecuentes (suele reducir el problema)

In [37]:
frequent_itemsets = fpgrowth(df, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.8)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Kidney Beans),(Eggs),1.0,0.8,0.8,0.8,1.00,1.0,0.00,1.0,0.0,0.80,0.0,0.900
1,(Eggs),(Kidney Beans),0.8,1.0,0.8,1.0,1.00,1.0,0.00,inf,0.0,0.80,0.0,0.900
2,(Yogurt),(Kidney Beans),0.6,1.0,0.6,1.0,1.00,1.0,0.00,inf,0.0,0.60,0.0,0.800
3,"(Yogurt, Eggs)",(Kidney Beans),0.4,1.0,0.4,1.0,1.00,1.0,0.00,inf,0.0,0.40,0.0,0.700
4,(Onion),(Eggs),0.6,0.8,0.6,1.0,1.25,1.0,0.12,inf,0.5,0.75,1.0,0.875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625,"(Corn, Eggs)","(Kidney Beans, Ice cream, Onion)",0.2,0.2,0.2,1.0,5.00,1.0,0.16,inf,1.0,1.00,1.0,1.000
626,"(Ice cream, Onion)","(Kidney Beans, Corn, Eggs)",0.2,0.2,0.2,1.0,5.00,1.0,0.16,inf,1.0,1.00,1.0,1.000
627,"(Kidney Beans, Ice cream)","(Onion, Corn, Eggs)",0.2,0.2,0.2,1.0,5.00,1.0,0.16,inf,1.0,1.00,1.0,1.000
628,"(Ice cream, Eggs)","(Onion, Kidney Beans, Corn)",0.2,0.2,0.2,1.0,5.00,1.0,0.16,inf,1.0,1.00,1.0,1.000
